In [ ]:
import csv

In [ ]:
import sqlite3

In [ ]:
con = sqlite3.connect("superstoreData.db")   # creating a database connection using sqlite 
cur = con.cursor()   # started a cursor so that work in this database can be done

In [ ]:
import pandas as pd

In [ ]:
%load_ext sql

In [ ]:
!pip install -q pandas==1.1.5

In [ ]:
%sql sqlite:///SuperstoreData.db

In [ ]:
import pandas
df = pandas.read_csv("Superstore_Dataset.csv", encoding='ISO-8859-1')
df.to_sql("Superstore_Data", con, if_exists='replace', index=False,method="multi")


## Getting information about table/dataset


In [ ]:
%sql SELECT name FROM sqlite_master WHERE type='table';

In [ ]:
%sql SELECT * FROM PRAGMA_TABLE_INFO('Superstore_DATA');

In [ ]:
%sql SELECT count(name) FROM PRAGMA_TABLE_INFO('Superstore_Data');

## removing duplicates from table


In [ ]:
%sql select count(*) from Superstore_data;

In [ ]:
%%sql 
CREATE TABLE new_superstore_table AS
SELECT DISTINCT * FROM Superstore_Data;


In [ ]:
%sql select count(*) from new_Superstore_table;

## Adding month and year column using order date 

In [ ]:
%sql select count(*) from new_Superstore_table;

In [ ]:
%%sql
-- Add new columns for day, month, and year
ALTER TABLE new_Superstore_table
ADD COLUMN day INTEGER;

ALTER TABLE new_Superstore_table
ADD COLUMN month INTEGER;

ALTER TABLE new_Superstore_table
ADD COLUMN year INTEGER;



In [ ]:
%%sql
-- Update the new columns with day, month, and year values
UPDATE new_Superstore_table
SET 
    
    year = CAST(substr("Order Date", -4) AS INTEGER);

In [ ]:
%%sql
-- Update the new columns with day, month, and year values
UPDATE new_Superstore_table
SET 
    
    month = CAST(substr(replace("Order Date", '/', ' '), 1, 2) AS INTEGER);

In [ ]:
%%sql 
select * from new_Superstore_table


## Extracting and Visualising data

In [ ]:
%%sql
select year , sum(sales) from new_Superstore_table group by year;
select year , month , sum(sales) from new_Superstore_table group by year , month;

In [ ]:
!pip install seaborn
import seaborn as sns

## Monthly Sales Trend

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Connect to the SQLite database
conn = sqlite3.connect('SuperstoreData.db')  # Adjust the database name if needed

# Execute the SQL query
query = '''
SELECT year, month, SUM(sales) as total_sales
FROM new_Superstore_table
GROUP BY year, month;
'''
result = pd.read_sql_query(query, conn)

# Close the database connection
conn.close()
result['date'] = pd.to_datetime(result[['year', 'month']].assign(day=1))

# Plotting using Seaborn
plt.figure(figsize=(12, 6))
sns.lineplot(data=result, x='date', y='total_sales', hue='year', marker='o', palette='viridis')
plt.title('Monthly Sales Trend')
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.xticks(rotation=45)
plt.grid(True)
plt.show()


### Sales by Category


In [ ]:
%%sql 
select category, sum(sales) as total_sales from new_superstore_table group by category order by total_sales;

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  # Don't forget to import seaborn

# Connect to the SQLite database
conn = sqlite3.connect('SuperstoreData.db')  # Adjust the database name if needed

# Execute the SQL query
query = '''
SELECT category, SUM(sales) AS total_sales
FROM new_superstore_table
GROUP BY category
ORDER BY total_sales;
'''
result = pd.read_sql_query(query, conn)
print(result.columns)



In [ ]:
# Close the database connection
conn.close()
from matplotlib.ticker import FuncFormatter
# Plotting using Seaborn
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=result, x='Category', y='total_sales')  # Make sure column names match the case
plt.title('Sales by Category')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: '{:.0f}K'.format(x / 1000)))

plt.show()

### Sales by Region


In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  # Don't forget to import seaborn

# Connect to the SQLite database
conn = sqlite3.connect('SuperstoreData.db')  # Adjust the database name if needed

# Execute the SQL query
query = '''
SELECT region, SUM(sales) AS total_sales
FROM new_superstore_table
GROUP BY region
ORDER BY total_sales;
'''
result = pd.read_sql_query(query, conn)
print(result)
print(result.columns)

# Close the database connection
conn.close()
from matplotlib.ticker import FuncFormatter
# Plotting using Seaborn
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=result, x='Region', y='total_sales')  # Make sure column names match the case
plt.title('Sales by Region')
plt.show()

### Sales by sub-category


In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  # Don't forget to import seaborn

# Connect to the SQLite database
conn = sqlite3.connect('SuperstoreData.db')  # Adjust the database name if needed

# Execute the SQL query
query = '''
SELECT "sub-category", SUM(sales) AS total_sales
FROM new_superstore_table
GROUP BY "sub-category"
ORDER BY total_sales;
'''
result = pd.read_sql_query(query, conn)
print(result)
print(result.columns)

# Close the database connection
conn.close()
from matplotlib.ticker import FuncFormatter
# Plotting using Seaborn
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=result, x='Sub-Category', y='total_sales')  # Make sure column names match the case
plt.title('Sales by Sub-Category')
plt.xticks(fontsize=7)

plt.show()

In [ ]:
# Monthly Profit trend
profit_by_month = data.groupby(['Order Year', 'Order Month'])['Profit'].sum().reset_index()

plt.figure(figsize=(12,6))
sns.lineplot(data=profit_by_month, x='Order Month', y='Profit', hue='Order Year')
plt.title('Montly Profit Trend')
plt.show()

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  # Don't forget to import seaborn

# Connect to the SQLite database
conn = sqlite3.connect('SuperstoreData.db')  # Adjust the database name if needed

# Execute the SQL query
query = '''
SELECT year,month, SUM(profit) AS total_profit
FROM new_superstore_table
GROUP BY year,month
ORDER BY year,month;
'''
result = pd.read_sql_query(query, conn)
print(result)
print(result.columns)

# Close the database connection
conn.close()
result['date'] = pd.to_datetime(result[['year', 'month']].assign(day=1))

result['cumulative_profit'] = result.groupby('year')['total_profit'].cumsum()

# Plotting using Seaborn
plt.figure(figsize=(12, 6))
sns.lineplot(data=result, x='month', y='cumulative_profit', hue='year', marker='o', palette='viridis')
plt.title('Cumulative Profit Trend by Year')
plt.xlabel('Date')
plt.ylabel('Cumulative Profit')
plt.xticks(rotation=45)
plt.grid(True)
plt.legend(title='Year', loc='upper left', bbox_to_anchor=(1, 1))

plt.show()

### Profit by category

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  # Don't forget to import seaborn

# Connect to the SQLite database
conn = sqlite3.connect('SuperstoreData.db')  # Adjust the database name if needed

# Execute the SQL query
query = '''
SELECT category, SUM(profit) AS total_profit
FROM new_superstore_table
GROUP BY category
ORDER BY total_profit;
'''
result = pd.read_sql_query(query, conn)
print(result.columns)
# Close the database connection
conn.close()
from matplotlib.ticker import FuncFormatter
# Plotting using Seaborn
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=result, x='Category', y='total_profit')  # Make sure column names match the case
plt.title('profit by category')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: '{:.0f}K'.format(x / 1000)))

plt.show()

### Profit by sub category

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  # Don't forget to import seaborn

# Connect to the SQLite database
conn = sqlite3.connect('SuperstoreData.db')  # Adjust the database name if needed

# Execute the SQL query
query = '''
SELECT "sub-category", SUM(profit) AS total_profit
FROM new_superstore_table
GROUP BY "sub-category"
ORDER BY total_profit;
'''
result = pd.read_sql_query(query, conn)
print(result)
print(result.columns)

# Close the database connection

conn.close()
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=result, x='Sub-Category', y='total_profit', palette='viridis')

plt.title('Profit by Sub-Category')
plt.xticks(fontsize=7)

plt.show()



### Profit by region

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  # Don't forget to import seaborn

# Connect to the SQLite database
conn = sqlite3.connect('SuperstoreData.db')  # Adjust the database name if needed

# Execute the SQL query
query = '''
SELECT region, SUM(profit) AS total_profit
FROM new_superstore_table
GROUP BY region
ORDER BY total_profit;
'''
result = pd.read_sql_query(query, conn)
print(result)
print(result.columns)

# Close the database connection
conn.close()
from matplotlib.ticker import FuncFormatter
# Plotting using Seaborn
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=result, x='Region', y='total_profit')  # Make sure column names match the case
plt.title('Profit by Region')
plt.show()

In [ ]:
import sqlite3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Connect to the SQLite database
conn = sqlite3.connect('SuperstoreData.db')  # Adjust the database name if needed

# Execute the SQL query to fetch the required columns
query = '''
SELECT profit, discount, quantity, sales, "postal code", "Row ID"
FROM new_superstore_table;
'''
data = pd.read_sql_query(query, conn)

# Close the database connection
conn.close()

# Calculate the correlation matrix
correlation_matrix = data.corr()

# Plot the correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix')
plt.show()
